# Information-Theoretic Intent Reconstruction and Causal Counterfactual Discovery
## Neural Marked Temporal Point Processes and Structural Causal Models

End-to-end research implementation covering data acquisition, order-book/event preprocessing, temporal features, intent representation learning, constrained counterfactual replay, intent–impact fusion, explanations, calibration, and walk-forward evaluation.

**Important:** the Kaggle dataset schema may differ from exchange-native event logs. This notebook inspects the available files and columns, maps common field names where possible, and explicitly marks proxy labels/simplified replay when the source lacks verified incident labels or a matching-engine event stream. Do not interpret proxy predictions as proof of manipulation or as identified causal effects.

## 1. Environment and configuration
Run this notebook from top to bottom. For Kaggle, authenticate/download the dataset as required by your environment. If the dataset is already downloaded, set `DATA_DIR` to its folder.

In [ ]:
%pip -q install pandas numpy scikit-learn scipy matplotlib seaborn torch captum statsmodels kagglehub

In [ ]:
import os, re, glob, json, math, random, warnings, time, platform
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (precision_score, recall_score, f1_score, average_precision_score,
                             roc_auc_score, confusion_matrix, classification_report,
                             brier_score_loss, mean_absolute_error, mean_squared_error)
from sklearn.calibration import calibration_curve
from sklearn.cluster import KMeans
from scipy.special import softmax
from scipy.stats import wilcoxon
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
save_dir = "Finance 4"
os.makedirs(save_dir, exist_ok=True)

# Option A: point to a local extracted dataset folder.
DATA_DIR = None
# Option B: download via KaggleHub (requires Kaggle access/configuration).
if DATA_DIR is None:
    try:
        import kagglehub
        DATA_DIR = kagglehub.dataset_download("rajsharg/lob-dataset")
        print("Downloaded dataset:", DATA_DIR)
    except Exception as e:
        print("KaggleHub download was not successful. Set DATA_DIR to the extracted dataset folder.\n", repr(e))
        DATA_DIR = None

print("Device:", DEVICE, "| Save directory:", os.path.abspath(save_dir))

## 2. Stage 1 — Market data acquisition and schema inspection

In [ ]:
if DATA_DIR is None:
    raise FileNotFoundError("Set DATA_DIR to the extracted Kaggle LOB dataset directory, then rerun this cell.")

root = Path(DATA_DIR)
files = [p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in {".csv", ".parquet", ".feather", ".txt"}]
print(f"Found {len(files)} tabular files")
for p in files[:40]:
    print(p.relative_to(root), f"({p.stat().st_size/1e6:.1f} MB)")
if not files:
    raise FileNotFoundError(f"No CSV/Parquet/Feather/TXT files found under {root}")

In [ ]:
def read_table(path, nrows=None):
    ext = path.suffix.lower()
    if ext == ".csv": return pd.read_csv(path, nrows=nrows, low_memory=False)
    if ext == ".parquet": return pd.read_parquet(path)
    if ext == ".feather": return pd.read_feather(path)
    if ext == ".txt": return pd.read_csv(path, sep=None, engine="python", nrows=nrows)
    raise ValueError(ext)

# Load the largest tabular file by default; change DATA_FILE if the dataset has multiple roles.
DATA_FILE = max(files, key=lambda p: p.stat().st_size)
raw = read_table(DATA_FILE)
print("Selected:", DATA_FILE)
print("Shape:", raw.shape)
display(raw.head())
display(pd.DataFrame({"column": raw.columns, "dtype": raw.dtypes.astype(str), "missing": raw.isna().sum().values}))

### Column mapping and label policy

The mapper below recognizes common LOB/event column names. Review the printed mapping. If the dataset is a snapshot dataset rather than event-level submissions/cancellations/executions, the notebook can still run feature extraction and risk modelling, but true event reconstruction and causal replay are necessarily approximations. Labels are never silently treated as verified manipulation labels.

In [ ]:
def norm_col(s):
    return re.sub(r"[^a-z0-9]+", "", str(s).lower())

ALIASES = {
    "timestamp": ["timestamp","time","datetime","eventtime","date","ts","seconds"],
    "event_type": ["eventtype","type","action","operation","event","message_type","side_event"],
    "price": ["price","orderprice","tradeprice","midprice","mid_price","lastprice"],
    "size": ["size","quantity","qty","volume","ordersize","tradevolume"],
    "bid": ["bid","bestbid","bidprice","bid_price","bid1","bidprice1"],
    "ask": ["ask","bestask","askprice","ask_price","ask1","askprice1"],
    "bid_size": ["bidsize","bidsize1","bidvolume","bid_qty","bidquantity","biddepth"],
    "ask_size": ["asksize","asksize1","askvolume","ask_qty","askquantity","askdepth"],
    "label": ["label","target","class","y","manipulation","ismanipulation","anomaly","fraud"],
    "trader_id": ["traderid","participantid","userid","agentid","accountid"],
    "order_id": ["orderid","id","order_id"],
}
normalized = {norm_col(c): c for c in raw.columns}
COL = {}
for key, aliases in ALIASES.items():
    hit = next((normalized[a] for a in aliases if a in normalized), None)
    if hit is None:
        # partial-match fallback
        hit = next((c for c in raw.columns if any(a in norm_col(c) for a in aliases if len(a) >= 5)), None)
    COL[key] = hit
print("Proposed column mapping:")
display(pd.DataFrame({"role": list(COL), "column": [COL[k] for k in COL]}))

In [ ]:
df = raw.copy()
# Timestamp conversion; preserve row order if no recognizable timestamp exists.
if COL["timestamp"] is not None:
    ts = pd.to_datetime(df[COL["timestamp"]], errors="coerce", utc=True)
    if ts.notna().mean() > 0.5:
        df["_time"] = ts
        df = df.sort_values("_time", kind="stable").reset_index(drop=True)
    else:
        num = pd.to_numeric(df[COL["timestamp"]], errors="coerce")
        df["_time"] = num
        df = df.sort_values("_time", kind="stable").reset_index(drop=True)
else:
    df["_time"] = np.arange(len(df), dtype=float)

before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print("Removed exact duplicate rows:", before-len(df))

# Numeric parsing for mapped market fields.
for key in ["price","size","bid","ask","bid_size","ask_size"]:
    c = COL[key]
    if c is not None:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# Canonical event-type field; infer only a coarse proxy if no event type exists.
if COL["event_type"] is not None:
    df["_event"] = df[COL["event_type"]].astype(str).str.lower().str.strip()
else:
    df["_event"] = "snapshot_or_unknown"

# Numeric source label, if present. Preserve original values and make a binary mapping only when sensible.
LABEL_SOURCE = COL["label"]
df["_label"] = np.nan
if LABEL_SOURCE is not None:
    s = df[LABEL_SOURCE]
    if pd.api.types.is_numeric_dtype(s):
        vals = sorted(pd.Series(s.dropna().unique()).tolist())
        if len(vals) == 2:
            df["_label"] = s.map({vals[0]:0, vals[1]:1})
        elif set(vals).issubset({0,1}):
            df["_label"] = s
    else:
        st = s.astype(str).str.lower().str.strip()
        positive = st.str.contains(r"manip|spoof|layer|abnormal|anomal|suspicious|fraud|1|true", regex=True)
        negative = st.str.contains(r"normal|legit|benign|0|false", regex=True)
        df.loc[negative & ~positive, "_label"] = 0
        df.loc[positive & ~negative, "_label"] = 1

print("Rows:", len(df), "| mapped labels:", int(df["_label"].notna().sum()),
      "| label distribution:", df["_label"].value_counts(dropna=False).to_dict())

## 3. Stage 2 — Feature engineering and compact temporal market representation

In [ ]:
def numeric_col(key, default=np.nan):
    c = COL.get(key)
    return pd.to_numeric(df[c], errors="coerce") if c is not None else pd.Series(default, index=df.index, dtype=float)

df["price_num"] = numeric_col("price")
df["size_num"] = numeric_col("size")
df["bid_num"] = numeric_col("bid")
df["ask_num"] = numeric_col("ask")
df["bid_size_num"] = numeric_col("bid_size", 0.0)
df["ask_size_num"] = numeric_col("ask_size", 0.0)

# Derive top-of-book proxies where possible.
df["mid_price"] = (df["bid_num"] + df["ask_num"]) / 2
df["spread"] = df["ask_num"] - df["bid_num"]
df["queue_imbalance"] = (df["bid_size_num"] - df["ask_size_num"]) / (
    df["bid_size_num"].abs() + df["ask_size_num"].abs() + 1e-8)

# Prefer observed price as mid-price fallback.
df["mid_price"] = df["mid_price"].where(df["mid_price"].notna(), df["price_num"])
df["price_return"] = df["mid_price"].pct_change().replace([np.inf,-np.inf], np.nan)
df["price_displacement"] = df["mid_price"].diff()
df["depth_total"] = df["bid_size_num"].fillna(0) + df["ask_size_num"].fillna(0)
df["depth_change"] = df["depth_total"].diff()
df["size_change"] = df["size_num"].diff()
df["event_gap"] = pd.to_numeric(df["_time"], errors="coerce").diff()
if not pd.api.types.is_datetime64_any_dtype(df["_time"]):
    df["event_gap"] = df["event_gap"].clip(lower=0)
else:
    df["event_gap"] = df["event_gap"].dt.total_seconds()

event_text = df["_event"].astype(str).str.lower()
df["is_cancel"] = event_text.str.contains("cancel|delete|remove").astype(float)
df["is_submit"] = event_text.str.contains("submit|add|new|limit").astype(float)
df["is_execute"] = event_text.str.contains("execut|trade|fill|match").astype(float)
df["cancel_intensity"] = df["is_cancel"].rolling(20, min_periods=1).mean()
df["execution_frequency"] = df["is_execute"].rolling(20, min_periods=1).mean()
df["liquidity_replenishment"] = df["depth_change"].rolling(10, min_periods=1).mean()

# Order-flow imbalance proxy; use bid/ask depth changes when present, otherwise signed size changes.
bid_delta = df["bid_size_num"].diff().fillna(0)
ask_delta = df["ask_size_num"].diff().fillna(0)
df["ofi"] = bid_delta - ask_delta
if df["ofi"].abs().sum() == 0:
    df["ofi"] = df["size_change"].fillna(0) * np.where(df["is_cancel"] > 0, -1, 1)

feature_cols = [
    "price_num","size_num","bid_num","ask_num","bid_size_num","ask_size_num",
    "mid_price","spread","queue_imbalance","price_return","price_displacement",
    "depth_total","depth_change","size_change","event_gap","cancel_intensity",
    "execution_frequency","liquidity_replenishment","ofi","is_cancel","is_submit","is_execute"
]
feature_cols = [c for c in feature_cols if c in df.columns and df[c].notna().any()]
X_raw = df[feature_cols].replace([np.inf,-np.inf], np.nan)
X_raw = X_raw.fillna(X_raw.median(numeric_only=True)).fillna(0)
# Fit imputation and scaling on the training period only to reduce temporal leakage.
n_rows = len(df)
train_cut = int(n_rows * .70)
train_medians = X_raw.iloc[:train_cut].median(numeric_only=True)
X_raw = X_raw.fillna(train_medians).fillna(0)
scaler = RobustScaler()
scaler.fit(X_raw.iloc[:train_cut])
X = scaler.transform(X_raw).astype(np.float32)
print("Feature matrix:", X.shape, "| Features:", feature_cols)

### Quick data-quality and market-state diagnostics

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes[0,0].plot(df["mid_price"].to_numpy()[:5000], lw=.8); axes[0,0].set_title("Mid/observed price (first 5,000 rows)")
axes[0,1].plot(df["spread"].to_numpy()[:5000], lw=.8); axes[0,1].set_title("Spread proxy")
sns.histplot(df["event_gap"].replace([np.inf,-np.inf],np.nan).dropna().clip(upper=df["event_gap"].quantile(.99)), ax=axes[1,0], bins=40)
axes[1,0].set_title("Event-time gap (clipped at 99th percentile)")
sns.heatmap(pd.DataFrame(X, columns=feature_cols).corr(), cmap="vlag", center=0, ax=axes[1,1])
axes[1,1].set_title("Feature correlation")
plt.tight_layout()
plt.savefig(os.path.join(save_dir, "stage2_market_diagnostics.png"), dpi=300, bbox_inches="tight")
plt.show()

## 4. Leakage-controlled temporal partition

Rows are split chronologically. A label is considered usable only when the source contains a recognizable binary label. If no usable labels exist, the notebook creates **proxy weak labels** from extreme rolling market-feature deviations solely to exercise the pipeline; those outputs must not be reported as verified manipulation detection results.

In [ ]:
n = len(df)
if n < 100:
    warnings.warn("Small dataset; metrics may be unstable.")
train_end, val_end = int(n*.70), int(n*.85)
idx_train, idx_val, idx_test = np.arange(0,train_end), np.arange(train_end,val_end), np.arange(val_end,n)

label_available = df["_label"].notna().sum() >= max(30, int(.05*n)) and df["_label"].dropna().nunique() == 2
if label_available:
    y = df["_label"].fillna(0).astype(int).to_numpy()
    LABEL_MODE = "source labels (inspect provenance before calling them verified)"
else:
    # Weak proxy labels based on training-period quantiles only; not ground truth.
    z = np.abs(pd.DataFrame(X).rolling(50, min_periods=5).mean().diff()).mean(axis=1).fillna(0).to_numpy()
    threshold = np.quantile(z[idx_train], .95) if len(idx_train) else np.quantile(z,.95)
    y = (z > threshold).astype(int)
    LABEL_MODE = "weak proxy labels (NOT verified manipulation labels)"
print("Label mode:", LABEL_MODE, "| class counts:", np.bincount(y, minlength=2))
print("Temporal split sizes:", len(idx_train), len(idx_val), len(idx_test))

## 5. Stage 3 — Information-theoretic intent reconstruction

A neural marked temporal point-process-style encoder combines event features, elapsed time, and event marks. A variational information bottleneck (VIB) maps sequence summaries to a stochastic latent code; a contrastive-style supervised representation objective encourages similar labels to cluster, and an intent-conditioned decoder reconstructs the next feature vector. This is a practical research baseline, not a claim that latent variables reveal a trader's actual mental state.

In [ ]:
SEQ_LEN = 20
BATCH_SIZE = 128
EPOCHS = 8
LR = 1e-3
LATENT_DIM = 16

class WindowDataset(Dataset):
    def __init__(self, X, y, indices, seq_len=20):
        self.X, self.y, self.indices, self.seq_len = X, y, np.asarray(indices), seq_len
        self.valid = self.indices[self.indices >= seq_len]
    def __len__(self): return len(self.valid)
    def __getitem__(self, k):
        end = int(self.valid[k]); start = end-self.seq_len
        return (torch.tensor(self.X[start:end], dtype=torch.float32),
                torch.tensor(self.y[end], dtype=torch.long),
                torch.tensor(self.X[end], dtype=torch.float32))

train_ds = WindowDataset(X, y, idx_train, SEQ_LEN)
val_ds = WindowDataset(X, y, idx_val, SEQ_LEN)
test_ds = WindowDataset(X, y, idx_test, SEQ_LEN)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

class IntentTppVIB(nn.Module):
    def __init__(self, n_features, hidden=64, latent=16, n_classes=2):
        super().__init__()
        self.temporal = nn.Sequential(
            nn.Conv1d(n_features, hidden, kernel_size=3, padding=1, groups=1),
            nn.GELU(),
            nn.Conv1d(hidden, hidden, kernel_size=5, padding=2, groups=1),
            nn.GELU()
        )
        self.gru = nn.GRU(hidden, hidden, batch_first=True)
        self.mu = nn.Linear(hidden, latent)
        self.logvar = nn.Linear(hidden, latent)
        self.intent_head = nn.Sequential(nn.Linear(latent, 32), nn.GELU(), nn.Linear(32,n_classes))
        self.decoder = nn.Sequential(nn.Linear(latent, 64), nn.GELU(), nn.Linear(64,n_features))
        self.time_head = nn.Sequential(nn.Linear(hidden, 32), nn.GELU(), nn.Linear(32,1), nn.Softplus())
    def forward(self, x):
        # x: batch x time x features
        h = self.temporal(x.transpose(1,2)).transpose(1,2)
        h, _ = self.gru(h)
        pooled = h[:,-1,:]
        mu, logvar = self.mu(pooled), self.logvar(pooled).clamp(-8,8)
        z = mu + torch.randn_like(mu) * torch.exp(.5*logvar) if self.training else mu
        return self.intent_head(z), self.decoder(z), mu, logvar, self.time_head(pooled)

model = IntentTppVIB(X.shape[1], latent=LATENT_DIM).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
class_counts = np.bincount(y[idx_train], minlength=2).astype(float)
class_weights = torch.tensor((class_counts.sum()/(2*np.maximum(class_counts,1))), dtype=torch.float32, device=DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights)

def run_epoch(loader, train=False, beta=1e-3, recon_weight=.25):
    model.train(train)
    losses, correct, total = [], 0, 0
    for xb, yb, next_x in loader:
        xb, yb, next_x = xb.to(DEVICE), yb.to(DEVICE), next_x.to(DEVICE)
        with torch.set_grad_enabled(train):
            logits, recon, mu, logvar, time_pred = model(xb)
            cls_loss = criterion(logits, yb)
            recon_loss = nn.functional.smooth_l1_loss(recon, next_x)
            kl = -.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
            loss = cls_loss + recon_weight*recon_loss + beta*kl
            if train:
                optimizer.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
        losses.append(loss.item()); correct += (logits.argmax(1)==yb).sum().item(); total += len(yb)
    return float(np.mean(losses)) if losses else np.nan, correct/max(total,1)

history = {"train_loss":[],"val_loss":[],"train_acc":[],"val_acc":[]}
for epoch in range(EPOCHS):
    tr_loss,tr_acc = run_epoch(train_loader, True)
    va_loss,va_acc = run_epoch(val_loader, False)
    history["train_loss"].append(tr_loss); history["val_loss"].append(va_loss)
    history["train_acc"].append(tr_acc); history["val_acc"].append(va_acc)
    print(f"Epoch {epoch+1:02d}/{EPOCHS} | train loss {tr_loss:.4f} acc {tr_acc:.3f} | val loss {va_loss:.4f} acc {va_acc:.3f}")

pd.DataFrame(history).to_csv(os.path.join(save_dir,"intent_training_history.csv"), index=False)
fig, ax = plt.subplots(1,2,figsize=(11,4))
ax[0].plot(history["train_loss"], label="Train"); ax[0].plot(history["val_loss"], label="Validation"); ax[0].set_title("Loss"); ax[0].legend()
ax[1].plot(history["train_acc"], label="Train"); ax[1].plot(history["val_acc"], label="Validation"); ax[1].set_title("Accuracy"); ax[1].legend()
plt.tight_layout(); plt.savefig(os.path.join(save_dir,"intent_model_history.png"),dpi=300,bbox_inches="tight"); plt.show()

In [ ]:
@torch.no_grad()
def collect_outputs(loader):
    model.eval()
    zs, probs, recons, labels, nextxs = [], [], [], [], []
    for xb,yb,nx in loader:
        xb=xb.to(DEVICE)
        logits,recon,mu,logvar,tau=model(xb)
        zs.append(mu.cpu().numpy()); probs.append(torch.softmax(logits,1).cpu().numpy())
        recons.append(recon.cpu().numpy()); labels.append(yb.numpy()); nextxs.append(nx.numpy())
    if not labels: return [np.empty((0,))]*5
    return tuple(np.concatenate(a,axis=0) for a in [zs,probs,recons,labels,nextxs])

Z_train,P_train,R_train,Y_train,N_train = collect_outputs(DataLoader(train_ds,batch_size=BATCH_SIZE))
Z_val,P_val,R_val,Y_val,N_val = collect_outputs(val_loader)
Z_test,P_test,R_test,Y_test,N_test = collect_outputs(test_loader)
print("Latent representation:", Z_test.shape, "test probabilities:", P_test.shape)

## 6. Stage 4 — Structural causal model and constrained counterfactual replay

This stage estimates an interpretable local structural model using lagged market variables, then creates a *feasible perturbation proxy* by modifying selected events/features and re-evaluating downstream outcomes. This is not a full exchange matching engine unless complete order-level events, order IDs, priority rules, and venue-specific semantics are available. Counterfactual differences are model-based estimates unless causal assumptions and replay fidelity are independently justified.

In [ ]:
# Build a compact causal-analysis table from available market variables.
causal_cols = [c for c in ["ofi","spread","depth_total","price_displacement","cancel_intensity",
                           "execution_frequency","queue_imbalance","price_return"] if c in df.columns]
causal_df = df[causal_cols].replace([np.inf,-np.inf],np.nan).ffill().fillna(0)
# Lagged regression: contemporaneous associations are not interpreted as causality.
target_causal = "price_displacement" if "price_displacement" in causal_df else causal_cols[0]
predictors = [c for c in causal_cols if c != target_causal]
lagged = pd.DataFrame(index=causal_df.index)
for c in predictors:
    lagged[c+"_lag1"] = causal_df[c].shift(1)
lagged[target_causal] = causal_df[target_causal]
causal_data = lagged.dropna()
split_c = max(1, int(.7*len(causal_data)))
Xc = causal_data.drop(columns=[target_causal]).to_numpy()
yc = causal_data[target_causal].to_numpy()
if Xc.shape[1] == 0:
    Xc = np.arange(len(yc)).reshape(-1,1)
from sklearn.linear_model import Ridge
scm = Ridge(alpha=1.0).fit(Xc[:split_c], yc[:split_c])
causal_pred = scm.predict(Xc)
causal_residual = yc - causal_pred
print("Lagged structural proxy R2 (held-out tail):",
      float(scm.score(Xc[split_c:],yc[split_c:])) if len(yc)-split_c > 1 else np.nan)
display(pd.DataFrame({"parent_feature": causal_data.drop(columns=[target_causal]).columns,
                      "coefficient": scm.coef_}))

In [ ]:
# Counterfactual proxy: neutralize OFI for a local window and estimate downstream price displacement.
# The intervention is performed in the lagged feature matrix; it is NOT an exchange-level order replay.
def counterfactual_impact(start, window=10, intervention_feature="ofi_lag1"):
    if intervention_feature not in causal_data.drop(columns=[target_causal]).columns:
        return {"status":"feature unavailable","observed":np.nan,"counterfactual":np.nan,"difference":np.nan}
    end = min(start+window, len(causal_data))
    X_obs = Xc[start:end].copy()
    X_cf = X_obs.copy()
    j = list(causal_data.drop(columns=[target_causal]).columns).index(intervention_feature)
    X_cf[:,j] = 0.0  # zero is the robust-scaled neutral point
    obs = scm.predict(X_obs)
    cf = scm.predict(X_cf)
    return {"status":"model-based counterfactual; no full matching-engine replay",
            "observed_predicted_impact":float(np.mean(obs)),
            "counterfactual_predicted_impact":float(np.mean(cf)),
            "difference":float(np.mean(obs-cf)), "window":int(end-start)}

cf_rows = []
stride = max(1, len(causal_data)//100)
for start in range(0, len(causal_data), stride):
    r = counterfactual_impact(start)
    r["start_index"] = start
    cf_rows.append(r)
counterfactuals = pd.DataFrame(cf_rows)
counterfactuals.to_csv(os.path.join(save_dir,"counterfactual_estimates.csv"), index=False)
display(counterfactuals.head())

### Bootstrap uncertainty intervals for estimated counterfactual differences

In [ ]:
diffs = counterfactuals["difference"].dropna().to_numpy()
if len(diffs):
    rng = np.random.default_rng(SEED)
    boots = np.array([np.mean(rng.choice(diffs, size=len(diffs), replace=True)) for _ in range(1000)])
    ci = np.quantile(boots,[.025,.975])
    print(f"Mean model-based counterfactual difference: {diffs.mean():.6g}; bootstrap 95% interval [{ci[0]:.6g}, {ci[1]:.6g}]")
    plt.figure(figsize=(7,4)); sns.histplot(diffs,bins=30); plt.axvline(0,ls="--")
    plt.title("Model-based counterfactual impact differences")
    plt.tight_layout(); plt.savefig(os.path.join(save_dir,"counterfactual_impact_distribution.png"),dpi=300,bbox_inches="tight"); plt.show()
else:
    print("No counterfactual differences available.")

## 7. Stage 5 — Intent–impact fusion and calibrated manipulation-risk detection

The fusion head combines latent intent features, estimated counterfactual impact summaries, and impact uncertainty. Class-balanced focal loss handles imbalance. Temperature scaling is fitted on the validation split. Conformal prediction uses a held-out calibration subset from the validation period and reports set coverage/size on the test period. If labels are weak proxies, all metrics are proxy-label metrics.

In [ ]:
# Align each sequence example with its row index in the temporal split.
def sample_end_indices(indices, seq_len=SEQ_LEN):
    return np.asarray(indices)[np.asarray(indices) >= seq_len]

train_ends = sample_end_indices(idx_train)
val_ends = sample_end_indices(idx_val)
test_ends = sample_end_indices(idx_test)

# Counterfactual impact feature aligned by row, with a lagged-regression impact proxy.
impact_all = np.zeros(len(df), dtype=np.float32)
impact_unc_all = np.zeros(len(df), dtype=np.float32)
if len(causal_data):
    # Map causal rows back to original row indices (lag shift drops initial rows).
    row_ids = causal_data.index.to_numpy()
    cf_lookup = {}
    for r in cf_rows:
        if np.isfinite(r.get("difference",np.nan)):
            cf_lookup[int(r["start_index"])] = r["difference"]
    for i, rid in enumerate(row_ids):
        impact_all[rid] = causal_residual[i] if i < len(causal_residual) else 0
    # Local residual volatility as an uncertainty proxy.
    impact_unc_all = pd.Series(impact_all).rolling(50,min_periods=5).std().fillna(0).to_numpy(dtype=np.float32)

def aligned_features(Z, ends):
    return np.column_stack([Z, impact_all[ends], impact_unc_all[ends]]).astype(np.float32)

F_train = aligned_features(Z_train, train_ends)
F_val = aligned_features(Z_val, val_ends)
F_test = aligned_features(Z_test, test_ends)
y_train = y[train_ends]; y_val = y[val_ends]; y_test = y[test_ends]
print(F_train.shape,F_val.shape,F_test.shape)

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__(); self.alpha=alpha; self.gamma=gamma
    def forward(self, logits, targets):
        logp=nn.functional.log_softmax(logits,dim=1)
        p=logp.exp()
        logpt=logp.gather(1,targets[:,None]).squeeze(1)
        pt=p.gather(1,targets[:,None]).squeeze(1)
        loss=-(1-pt).pow(self.gamma)*logpt
        if self.alpha is not None: loss=loss*self.alpha[targets]
        return loss.mean()

class FusionNet(nn.Module):
    def __init__(self, d, hidden=48):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(d,hidden),nn.GELU(),nn.Dropout(.15),
                               nn.Linear(hidden,hidden//2),nn.GELU(),nn.Linear(hidden//2,2))
    def forward(self,x): return self.net(x)

fusion = FusionNet(F_train.shape[1]).to(DEVICE)
counts=np.bincount(y_train,minlength=2).astype(float)
alpha=torch.tensor(counts.sum()/(2*np.maximum(counts,1)),dtype=torch.float32,device=DEVICE)
focal=FocalLoss(alpha=alpha)
opt=torch.optim.AdamW(fusion.parameters(),lr=1e-3,weight_decay=1e-4)

def as_loader(F,Y,shuffle=False):
    ds=torch.utils.data.TensorDataset(torch.tensor(F,dtype=torch.float32),torch.tensor(Y,dtype=torch.long))
    return DataLoader(ds,batch_size=256,shuffle=shuffle)

fusion_hist={"train_loss":[],"val_loss":[]}
for ep in range(12):
    fusion.train(); tl=[]
    for xb,yb in as_loader(F_train,y_train,True):
        xb,yb=xb.to(DEVICE),yb.to(DEVICE)
        opt.zero_grad(); logits=fusion(xb); loss=focal(logits[:,:2],yb)
        loss.backward(); nn.utils.clip_grad_norm_(fusion.parameters(),1.0); opt.step(); tl.append(loss.item())
    fusion.eval(); vl=[]
    with torch.no_grad():
        for xb,yb in as_loader(F_val,y_val):
            xb,yb=xb.to(DEVICE),yb.to(DEVICE)
            vl.append(focal(fusion(xb)[:,:2],yb).item())
    fusion_hist["train_loss"].append(np.mean(tl)); fusion_hist["val_loss"].append(np.mean(vl) if vl else np.nan)
    print(f"Epoch {ep+1:02d}/12 | train {fusion_hist['train_loss'][-1]:.4f} | val {fusion_hist['val_loss'][-1]:.4f}")

pd.DataFrame(fusion_hist).to_csv(os.path.join(save_dir,"fusion_training_history.csv"),index=False)
plt.figure(figsize=(7,4)); plt.plot(fusion_hist["train_loss"],label="Train"); plt.plot(fusion_hist["val_loss"],label="Validation")
plt.title("Intent–impact fusion training"); plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(save_dir,"fusion_training_history.png"),dpi=300,bbox_inches="tight"); plt.show()

In [ ]:
@torch.no_grad()
def predict_logits(F):
    fusion.eval()
    out=[]
    for xb,_ in as_loader(F,np.zeros(len(F),dtype=int)):
        out.append(fusion(xb.to(DEVICE))[:,:2].cpu().numpy())
    return np.concatenate(out) if out else np.empty((0,2))

val_logits = predict_logits(F_val)
test_logits = predict_logits(F_test)

# Temperature scaling optimized on validation labels.
temperature = torch.nn.Parameter(torch.ones(1,device=DEVICE))
if len(val_logits):
    opt_t=torch.optim.LBFGS([temperature],lr=.05,max_iter=50)
    lv=torch.tensor(val_logits,dtype=torch.float32,device=DEVICE)
    yv=torch.tensor(y_val,dtype=torch.long,device=DEVICE)
    def closure():
        opt_t.zero_grad()
        loss=nn.functional.cross_entropy(lv/temperature.clamp(.05,10),yv)
        loss.backward()
        return loss
    opt_t.step(closure)
T=float(temperature.detach().clamp(.05,10).cpu())
test_prob=softmax(test_logits/T,axis=1)[:,1] if len(test_logits) else np.array([])
test_pred=(test_prob>=.5).astype(int)
print("Validation-fitted temperature:",T)

In [ ]:
def expected_calibration_error(y_true, prob, n_bins=10):
    if len(prob)==0: return np.nan
    edges=np.linspace(0,1,n_bins+1); ece=0.0
    for lo,hi in zip(edges[:-1],edges[1:]):
        mask=(prob>=lo)&(prob<(hi if hi<1 else hi+1e-9))
        if mask.any(): ece += mask.mean()*abs(y_true[mask].mean()-prob[mask].mean())
    return float(ece)

def safe_auc(y_true, prob):
    return roc_auc_score(y_true,prob) if len(np.unique(y_true))==2 else np.nan

metrics = {
    "label_mode": LABEL_MODE,
    "precision": precision_score(y_test,test_pred,zero_division=0) if len(y_test) else np.nan,
    "recall": recall_score(y_test,test_pred,zero_division=0) if len(y_test) else np.nan,
    "macro_f1": f1_score(y_test,test_pred,average="macro",zero_division=0) if len(y_test) else np.nan,
    "pr_auc": average_precision_score(y_test,test_prob) if len(y_test) and len(np.unique(y_test))==2 else np.nan,
    "roc_auc": safe_auc(y_test,test_prob) if len(y_test) else np.nan,
    "false_positive_rate": (np.sum((y_test==0)&(test_pred==1))/max(np.sum(y_test==0),1)) if len(y_test) else np.nan,
    "brier_score": brier_score_loss(y_test,test_prob) if len(y_test) else np.nan,
    "ece": expected_calibration_error(y_test,test_prob) if len(y_test) else np.nan,
    "temperature": T,
}
display(pd.DataFrame([metrics]))
pd.DataFrame([metrics]).to_csv(os.path.join(save_dir,"test_metrics.csv"),index=False)
if len(y_test):
    print(classification_report(y_test,test_pred,zero_division=0))
    cm=confusion_matrix(y_test,test_pred,labels=[0,1])
    sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",xticklabels=["Pred 0","Pred 1"],yticklabels=["True 0","True 1"])
    plt.title("Test confusion matrix"); plt.tight_layout()
    plt.savefig(os.path.join(save_dir,"test_confusion_matrix.png"),dpi=300,bbox_inches="tight"); plt.show()
    if len(np.unique(y_test))==2:
        frac_pos,mean_pred=calibration_curve(y_test,test_prob,n_bins=10,strategy="quantile")
        plt.figure(figsize=(5,5)); plt.plot(mean_pred,frac_pos,"o-",label="Temperature-scaled")
        plt.plot([0,1],[0,1],"--",label="Ideal"); plt.xlabel("Mean predicted probability"); plt.ylabel("Observed positive fraction")
        plt.title("Calibration curve"); plt.legend(); plt.tight_layout()
        plt.savefig(os.path.join(save_dir,"calibration_curve.png"),dpi=300,bbox_inches="tight"); plt.show()

### Conformal prediction sets

Use a calibration subset drawn from the validation period. The nonconformity score is `1 - p(true class)`. Coverage is measured on the later test period; finite-sample guarantees require exchangeability, which temporal market data often violates, so treat coverage as empirical and monitor it across regimes.

In [ ]:
if len(F_val) >= 20 and len(F_test):
    rng=np.random.default_rng(SEED)
    order=rng.permutation(len(F_val))
    cal_idx=order[:max(10,len(F_val)//2)]
    cal_prob=softmax(val_logits[cal_idx]/T,axis=1)
    cal_y=y_val[cal_idx]
    scores=1-cal_prob[np.arange(len(cal_y)),cal_y]
    alpha_cp=.10
    q=np.quantile(scores, min(1.0, math.ceil((len(scores)+1)*(1-alpha_cp))/len(scores)), method="higher")
    test_prob2=softmax(test_logits/T,axis=1)
    sets=[np.where(1-p<=q)[0].tolist() for p in test_prob2]
    coverage=np.mean([int(y_test[i] in sets[i]) for i in range(len(y_test))])
    avg_size=np.mean([len(s) for s in sets])
    print({"nominal_error":alpha_cp,"threshold":float(q),"empirical_coverage":float(coverage),"mean_set_size":float(avg_size)})
    pd.DataFrame({"true_label":y_test,"p0":test_prob2[:,0],"p1":test_prob2[:,1],
                  "prediction_set":[",".join(map(str,s)) for s in sets]}).to_csv(os.path.join(save_dir,"conformal_predictions.csv"),index=False)
else:
    print("Not enough validation/test examples for a useful conformal calibration split.")

## 8. Stage 6 — Explainability, ablation, and counterfactual consistency checks

Integrated Gradients is applied to the fusion model's positive-class logit when Captum is available. Event-level feature ablation provides a simple independent sensitivity check. The explanation is evidence about model behaviour, not evidence of intent or legal wrongdoing.

In [ ]:
# Integrated Gradients for fusion features.
try:
    from captum.attr import IntegratedGradients
    fusion.eval()
    sample_n=min(256,len(F_test))
    xt=torch.tensor(F_test[:sample_n],dtype=torch.float32,device=DEVICE,requires_grad=True)
    baseline=torch.zeros_like(xt)
    ig=IntegratedGradients(lambda inp: fusion(inp)[:,1])
    attr=ig.attribute(xt,baselines=baseline,n_steps=32)
    importance=attr.abs().mean(0).detach().cpu().numpy()
except Exception as e:
    print("Integrated Gradients unavailable; using permutation-style feature ablation:", repr(e))
    importance=np.zeros(F_test.shape[1])
    if len(F_test):
        base=f1_score(y_test,(softmax(test_logits/T,axis=1)[:,1]>=.5).astype(int),zero_division=0)
        for j in range(F_test.shape[1]):
            perm=F_test.copy(); perm[:,j]=np.median(F_train[:,j])
            p=softmax(predict_logits(perm)/T,axis=1)[:,1]
            importance[j]=base-f1_score(y_test,(p>=.5).astype(int),zero_division=0)

explain_names=[f"intent_latent_{i}" for i in range(LATENT_DIM)]+["counterfactual_impact_proxy","impact_uncertainty_proxy"]
explain_names=explain_names[:len(importance)]
explain_df=pd.DataFrame({"feature":explain_names,"importance":importance}).sort_values("importance",ascending=False)
display(explain_df)
explain_df.to_csv(os.path.join(save_dir,"feature_importance.csv"),index=False)
plt.figure(figsize=(8,5)); sns.barplot(data=explain_df.head(15),x="importance",y="feature")
plt.title("Fusion model feature attribution"); plt.tight_layout()
plt.savefig(os.path.join(save_dir,"feature_attribution.png"),dpi=300,bbox_inches="tight"); plt.show()

In [ ]:
# Event-level feature ablation on the original input representation.
# For each feature, replace it with its training median and measure the intent classifier F1 change.
def intent_predict_with_X(X_input, indices, batch_size=256):
    # windows ending at supplied indices
    model.eval(); out=[]
    with torch.no_grad():
        for e in indices:
            if e < SEQ_LEN: continue
            xb=torch.tensor(X_input[e-SEQ_LEN:e][None],dtype=torch.float32,device=DEVICE)
            logits,*_=model(xb); out.append(torch.softmax(logits,1).cpu().numpy()[0])
    return np.asarray(out)

baseline_prob=intent_predict_with_X(X,test_ends)
baseline_f1=f1_score(y_test,baseline_prob.argmax(1),average="macro",zero_division=0) if len(baseline_prob) else np.nan
ablation=[]
for j,name in enumerate(feature_cols):
    Xa=X.copy()
    Xa[:,j]=np.median(X[idx_train,j])
    pp=intent_predict_with_X(Xa,test_ends)
    score=f1_score(y_test,pp.argmax(1),average="macro",zero_division=0) if len(pp) else np.nan
    ablation.append({"feature":name,"baseline_macro_f1":baseline_f1,"ablated_macro_f1":score,
                     "f1_drop":baseline_f1-score if np.isfinite(score) and np.isfinite(baseline_f1) else np.nan})
ablation_df=pd.DataFrame(ablation).sort_values("f1_drop",ascending=False)
display(ablation_df.head(15))
ablation_df.to_csv(os.path.join(save_dir,"event_feature_ablation.csv"),index=False)

## 9. Stage 7 — Walk-forward robustness, regime analysis, and computational profile

In [ ]:
# Walk-forward evaluation using the trained test predictions grouped by sequential chunks.
# This is a descriptive fold analysis; it does not retrain the full deep model in every fold.
fold_rows=[]
if len(y_test) >= 10:
    chunks=np.array_split(np.arange(len(y_test)), min(5,max(2,len(y_test)//10)))
    for k,ix in enumerate(chunks,1):
        if len(ix)==0: continue
        fold_rows.append({
            "fold":k,"n":len(ix),
            "precision":precision_score(y_test[ix],test_pred[ix],zero_division=0),
            "recall":recall_score(y_test[ix],test_pred[ix],zero_division=0),
            "macro_f1":f1_score(y_test[ix],test_pred[ix],average="macro",zero_division=0),
            "pr_auc":average_precision_score(y_test[ix],test_prob[ix]) if len(np.unique(y_test[ix]))==2 else np.nan,
            "brier":brier_score_loss(y_test[ix],test_prob[ix])
        })
fold_df=pd.DataFrame(fold_rows)
display(fold_df)
fold_df.to_csv(os.path.join(save_dir,"walk_forward_test_chunks.csv"),index=False)

# Cross-regime stress test using training-period volatility/depth quantiles.
vol=df["price_return"].rolling(50,min_periods=5).std().fillna(0).to_numpy()
depth=df["depth_total"].fillna(0).to_numpy()
vol_cut=np.quantile(vol[idx_train],[.33,.67]) if len(idx_train) else [0,0]
regime=np.where(vol<=vol_cut[0],"low_volatility",np.where(vol<=vol_cut[1],"mid_volatility","high_volatility"))
regime_test=regime[test_ends]
regime_rows=[]
for r in np.unique(regime_test):
    ix=np.where(regime_test==r)[0]
    regime_rows.append({"regime":r,"n":len(ix),
                        "precision":precision_score(y_test[ix],test_pred[ix],zero_division=0),
                        "recall":recall_score(y_test[ix],test_pred[ix],zero_division=0),
                        "macro_f1":f1_score(y_test[ix],test_pred[ix],average="macro",zero_division=0)})
regime_df=pd.DataFrame(regime_rows)
display(regime_df)
regime_df.to_csv(os.path.join(save_dir,"cross_regime_results.csv"),index=False)

In [ ]:
# Runtime and memory profile (parameter count plus simple batch inference timing).
n_params=sum(p.numel() for p in model.parameters())+sum(p.numel() for p in fusion.parameters())
start=time.perf_counter()
if len(F_test):
    _=predict_logits(F_test[:min(1024,len(F_test))])
elapsed=time.perf_counter()-start
profile={
    "device":DEVICE,
    "parameter_count_total":int(n_params),
    "fusion_inference_seconds_for_n":float(elapsed),
    "fusion_inference_rows":int(min(1024,len(F_test))),
    "fusion_latency_ms_per_row":float(1000*elapsed/max(1,min(1024,len(F_test)))),
    "peak_cuda_memory_mb":float(torch.cuda.max_memory_allocated()/1024**2) if torch.cuda.is_available() else np.nan,
    "python_version":platform.python_version(),
    "torch_version":torch.__version__,
}
display(pd.DataFrame([profile]))
pd.DataFrame([profile]).to_csv(os.path.join(save_dir,"runtime_profile.csv"),index=False)

## 10. Component ablations and baseline comparison

This section reports comparable test-set baselines using the same temporal split. It includes conventional classifiers on the engineered features, intent-only output, impact-only output, and the combined fusion model. Since the full fusion net and intent model are not retrained for every ablation, treat the module ablations as diagnostic rather than definitive causal contribution estimates.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

# Row-level baseline uses engineered features at each test endpoint.
Xtr_base=X[train_ends]; Xv_base=X[val_ends]; Xte_base=X[test_ends]
baseline_models={
    "Logistic regression": LogisticRegression(max_iter=500,class_weight="balanced"),
    "Random forest": RandomForestClassifier(n_estimators=150,max_depth=10,class_weight="balanced_subsample",random_state=SEED,n_jobs=-1),
    "HistGradientBoosting": HistGradientBoostingClassifier(max_iter=150,random_state=SEED)
}
baseline_rows=[]
for name,clf in baseline_models.items():
    clf.fit(Xtr_base,y_train)
    pred=clf.predict(Xte_base)
    prob=clf.predict_proba(Xte_base)[:,1] if hasattr(clf,"predict_proba") else pred.astype(float)
    baseline_rows.append({"method":name,"precision":precision_score(y_test,pred,zero_division=0),
                          "recall":recall_score(y_test,pred,zero_division=0),
                          "macro_f1":f1_score(y_test,pred,average="macro",zero_division=0),
                          "pr_auc":average_precision_score(y_test,prob) if len(np.unique(y_test))==2 else np.nan})
# Intent-only
intent_pred=P_test.argmax(1) if len(P_test) else np.array([])
baseline_rows.append({"method":"Intent-only (TPP-VIB)","precision":precision_score(Y_test,intent_pred,zero_division=0),
                      "recall":recall_score(Y_test,intent_pred,zero_division=0),
                      "macro_f1":f1_score(Y_test,intent_pred,average="macro",zero_division=0),
                      "pr_auc":average_precision_score(Y_test,P_test[:,1]) if len(np.unique(Y_test))==2 else np.nan})
# Impact-only logistic regression
impact_only_train=F_train[:,-2:]; impact_only_test=F_test[:,-2:]
impact_clf=LogisticRegression(max_iter=500,class_weight="balanced").fit(impact_only_train,y_train)
ip=impact_clf.predict(impact_only_test); ipp=impact_clf.predict_proba(impact_only_test)[:,1]
baseline_rows.append({"method":"Impact-only","precision":precision_score(y_test,ip,zero_division=0),
                      "recall":recall_score(y_test,ip,zero_division=0),
                      "macro_f1":f1_score(y_test,ip,average="macro",zero_division=0),
                      "pr_auc":average_precision_score(y_test,ipp) if len(np.unique(y_test))==2 else np.nan})
baseline_rows.append({"method":"Proposed intent-impact fusion","precision":precision_score(y_test,test_pred,zero_division=0),
                      "recall":recall_score(y_test,test_pred,zero_division=0),
                      "macro_f1":f1_score(y_test,test_pred,average="macro",zero_division=0),
                      "pr_auc":average_precision_score(y_test,test_prob) if len(np.unique(y_test))==2 else np.nan})
comparison=pd.DataFrame(baseline_rows)
display(comparison)
comparison.to_csv(os.path.join(save_dir,"baseline_comparison.csv"),index=False)

## 11. Export artifacts and final audit

The notebook exports intermediate metrics, explanations, counterfactual estimates, and plots into `save_dir`. Before publication, confirm source dataset licensing, event semantics, label provenance, temporal splits, and whether the data supports genuine order-level replay. Do not describe weak proxy labels as verified manipulation labels.

In [ ]:
# Save a concise run manifest and model weights.
manifest = {
    "dataset_path": str(DATA_FILE),
    "dataset_shape": list(raw.shape),
    "feature_columns": feature_cols,
    "label_mode": LABEL_MODE,
    "split_sizes": {"train":int(len(idx_train)),"validation":int(len(idx_val)),"test":int(len(idx_test))},
    "sequence_length":SEQ_LEN,
    "latent_dim":LATENT_DIM,
    "epochs_intent":EPOCHS,
    "device":DEVICE,
    "notes":[
        "Inspect source label provenance before interpreting detection metrics.",
        "If weak proxy labels were used, metrics are not verified manipulation detection results.",
        "Counterfactual estimates are model-based unless matching-engine replay and causal assumptions are validated.",
        "Intent embeddings are behavioural representations, not direct observations of mental state."
    ]
}
with open(os.path.join(save_dir,"run_manifest.json"),"w",encoding="utf-8") as f:
    json.dump(manifest,f,indent=2)
torch.save({"intent_model":model.state_dict(),"fusion_model":fusion.state_dict(),
            "feature_cols":feature_cols,"temperature":T},
           os.path.join(save_dir,"models.pt"))
print("Completed. Artifacts saved to:", os.path.abspath(save_dir))
print("\n".join(sorted(os.listdir(save_dir))))